In [ ]:
import warnings
warnings.filterwarnings('ignore', '.*overflowing tokens.*')
import torch
from torch import nn
from transformer import BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW
import torch.nn.Functional as F
import pandas as pd
import numpy as np
def set_seeds(seed = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = ''
tokenizer = BertTokenizer.from_pretrained(model_path)
train_data = pd.read_csv()
test_data = pd.read_csv()
print('train_data is:' train_data.shape, 'test_data is:' test_data.shape)

def tokenizetexts(df, tokenizer, max_length = 512):
    encoding_a = tokenizer(
        df['response_a'],
        padding = 'max_length',
        truncation = True,
        max_length = max_length
        return_tensors = 'pt'
    )
    encoding_b = tokenizer(
        df['response_b'],
        padding = 'max_length',
        truncation = True,
        max_length = max_length,
        return_tensors = 'pt'
    )
    return {
        'input_ids_a': encoding_a['input_ids'],
        'attention_mask_a' : encoding_a['attention_mask'],
        'input_ids_b': encoding_b['input_ids'],
        'attention_mask_b': encoding_b['attention_mask'] 
    }
print('Processing data....') 
preprocessedtrain = tokenizetexts(train_data, tokenizer)
labels = torch.tensor(train_data[['winner_model_a', 'winner_model_b', 'winner_tie']].values, torch.float32)

#creating a dataset inorder to convert it into a dataloader to be pass in our model
class ChatbotDataset(Dataset):
    def __init__(self, preprocessed_data, labels=None):
        self.input_ids_a = preprocessed_data['input_ids_a']
        self.attention_mask_a = preprocessed_data['attention_mask_a']
        self.input_ids_b = preprocessed_data['input_ids_b']
        self.attention_mask_b = preprocessed_data['attention_mask_b']
        self.labels = labels if labels is not None else torch.zeros(len(self.input_ids_a), 3)

        def __len__(self):
            return self.input_ids_a

        def __getitem__(self, idx):
            batch = {
                'input_ids_a': self.input_ids_a[idx],
                'attention_mask_a': self.attention_mask_a[idx],
                'input_ids_b':self.input_ids_a[idx],
                'attention_mask_b': self.attention_mask_b[idx]
            }
            if labels is not None:
                batch['labels'] = self.labels[idx]
            return batch
train_dataset = ChatbotDataset(preprocessed_train, labels)
train_dataloader = Dataloader(train_dataset, batch_size = 32, shuffle = True num_workers = os.cpu_count(), pin_memory = True)

class MultiTaskBERTModel(nn.Module):
    def __init__(self, model_path, num_labels = 3):
        super(MultiTaskBERTModel, self).__init__()
        self.bert = BertForSequenceClassification(
            model_path,
            num_labels = num_labels,
            ignore_mismatched_sizes = True,
        )
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        nn.init.xavier_uniform_(self.bert.classifier.weights)
        nn.zeros(self.bert.classifier.bias)

    def forward(self, input_ids_a, attention_mask_a, input_ids_b, attention_mask_b):
        output_a = self.bert(input_ids_a, attention_mask_a).logits
        output_b = self.bert(input_ids_b, attention_mask_b).logits
        return output_a, output_b

model = MultiTaskBERTModel()
model.to(device)

print('Generating predictions...')
processed_test = tokenizetexts(test_data)
test_dataset = ChatbotDataset(processed_test, labels)
test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = False, num_workers = 4, pin_memory = True)

logits_a, logits_b = [], []
model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k,v in batch.items() if 'input_ids' in k or 'attention_mask' in k}
        out_a,out_b = model(**batch)
        logits_a.append(out_a.cpu())
        logits_b.append(out_b.cpu())

logits_a = torch.cat(logits_a, dim=0)
logits_b = torch.cat(logits_b, dim=0)
logits_tie = torch.zero_like(logits_a[:,0], 3)

logits = torch.stack([logits_a[:, 0], logits_b[:,0], logits_tie], dim=1)
probs = F.softmax(logits, dim = 1).numpy()

submission_df = pd.DataFrame({
    'id': test_data['id'].values,
    'winner_model_a': probs[:, 0],
    'winner_model_b': probs[:, 1],
    'winner_tie': probs[:, 2]
})
submission_df.to_csv('submission.csv', index = False)